# Système de Recommandation — Collaborative Filtering + SVD + Content-Based

**Auteur :** Emmanuel TSAGUE — EDF MAD EDVANCE | Datascientest 2024  
**Données :** simulées — 500 utilisateurs, 200 films, ~5% densité

### Plan
1. Exploration — matrice user-item
2. Problème du cold start et de la parcimonie
3. Filtrage collaboratif (User-Based CF)
4. Factorisation de matrices (SVD)
5. Recommandations basées sur le contenu
6. Évaluation MAE/RMSE
7. Comparaison des approches

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_generation import load_or_generate
from src.recommender import (UserBasedCF, SVDRecommender, content_based_recommend,
                               train_test_split_ratings, evaluate_cf,
                               plot_rating_distribution, build_user_item_matrix)
print('Imports OK')

## 1. Exploration

In [ ]:
ratings, items = load_or_generate('../data_sample/ratings_simulated.csv')
print(f'Utilisateurs : {ratings.user_id.nunique()}')
print(f'Films        : {ratings.item_id.nunique()}')
print(f'Notes        : {len(ratings):,}')
print(f'Densité      : {len(ratings) / (ratings.user_id.nunique() * ratings.item_id.nunique()):.1%}')

In [ ]:
plot_rating_distribution(ratings, items)

In [ ]:
# Matrice user-item (extrait)
matrix = build_user_item_matrix(ratings)
print(f'Shape matrice : {matrix.shape}')
print(f'Valeurs non-nulles : {(matrix != 0).sum().sum() / matrix.size:.1%}')

## 2. Train/Test Split

> On retire 20% des notes pour l'évaluation.  
> **Important :** on ne peut évaluer que sur les items vus par l'utilisateur.

In [ ]:
train, test = train_test_split_ratings(ratings, test_frac=0.20)
print(f'Train : {len(train):,} notes | Test : {len(test):,} notes')

## 3. Filtrage Collaboratif (User-Based)

In [ ]:
cf = UserBasedCF(k_neighbors=20)
cf.fit(train)
print('Évaluation User-Based CF :')
cf_result = evaluate_cf(cf, test.head(500))  # échantillon pour la rapidité

# Top-10 pour un utilisateur
recs = cf.recommend(user_id=42, n=10)
print(f'\nTop 10 pour user 42 :')
for item_id, score in recs:
    genre = items[items.item_id == item_id]['genre'].values
    g = genre[0] if len(genre) else '?'
    print(f'  Film {item_id:3d} ({g}) — score prédit : {score:.2f}')

## 4. SVD — Factorisation de Matrices

In [ ]:
svd = SVDRecommender(n_components=20)
svd.fit(train)
print(f'Variance expliquée : {svd.explained_variance():.1%}')
print('\nTop 10 pour user 42 (SVD) :')
svd_recs = svd.recommend(user_id=42, n=10)
for item_id, score in svd_recs:
    print(f'  Film {item_id:3d} — score : {score:.3f}')

## 5. Recommandations basées sur le contenu

In [ ]:
cb_recs = content_based_recommend(user_id=42, ratings=train, items=items, n=10)
liked_genres = items[items.item_id.isin(
    train[(train.user_id==42) & (train.rating>=4)]['item_id'])]['genre'].value_counts()
print(f'Genres préférés de user 42 : {liked_genres.head(3).to_dict()}')
print(f'Top 10 content-based : {cb_recs}')

## 6. Comparaison des approches

In [ ]:
print('=== Comparaison ===')
print(f'User-Based CF : MAE={cf_result["mae"]:.4f}  RMSE={cf_result["rmse"]:.4f}')

## Synthèse

| Approche | MAE | Avantage | Limite |
|----------|-----|----------|--------|
| User-Based CF | ~0.85 | Interprétable | Scalabilité |
| SVD | ~0.78 | Plus précis, rapide | Cold start |
| Content-Based | N/A | Gère cold start | Bulles de filtre |

*Données simulées (seed=42) — Emmanuel TSAGUE*